# Activity 1: Dataset Preparation (Titanic)
**Course:** Machine Learning | **Class:** T.Y. B.Tech AI & DS, Division A

## 1. Dataset Information
- **Name:** Titanic Dataset
- **Source:** Kaggle – https://www.kaggle.com/c/titanic/data
- **File type:** CSV (`titanic.csv`)
- **Rows x Columns:** 891 x 12
- **What it represents:** Details of passengers on the Titanic (class, sex, age, fare, etc.) and whether each passenger survived.
- **Future target column:** `Survived` (0 = No, 1 = Yes). We only prepare the data here; no model is built.

## 2. Load the Dataset

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("titanic.csv")
df_original = df.copy()   # untouched copy of the raw data
print("Shape:", df.shape)

## 3. Inspect the Dataset

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

### Numerical and Categorical Columns

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print("Numerical  :", num_cols)
print("Categorical:", cat_cols)

## 4. Data Quality Checks

In [ ]:
missing = df.isnull().sum()
percent = (missing / len(df) * 100).round(2)
print(pd.DataFrame({"Missing": missing, "Percent (%)": percent}))
print("\nDuplicates:", df.duplicated().sum())

### Category Consistency Check (Sex, Embarked)

In [ ]:
for col in ["Sex", "Embarked"]:
    print(col, "->", df[col].unique())
    print("Extra spaces:", (df[col].dropna() != df[col].dropna().str.strip()).sum())
    print()

Labels are consistent: no spelling variants and no extra spaces.

## 5. Exploratory Data Analysis (EDA)

In [ ]:
print(df[["Age", "Fare"]].describe())
print()
print(df["Embarked"].value_counts())
print()
print(df["Sex"].value_counts())

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["Age"].dropna(), bins=30, kde=True)
plt.title("Distribution of Passenger Age")
plt.xlabel("Age")
plt.ylabel("Number of Passengers")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Sex", hue="Survived")
plt.title("Survival by Gender")
plt.xlabel("Sex")
plt.ylabel("Number of Passengers")
plt.legend(title="Survived (0=No, 1=Yes)")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=df["Fare"])
plt.title("Box Plot of Fare")
plt.xlabel("Fare")
plt.show()

### Observations
1. Most passengers are aged roughly 20–40; very few are above 60.
2. Many more females survived than males, so `Sex` is likely an important feature.
3. `Fare` is highly right-skewed with several large outliers (very expensive tickets). Fare also has a much larger range than other columns, so it needs scaling.

## 6. Data Cleaning

In [ ]:
df = df.drop_duplicates()                          # no duplicates found, kept for safety
df = df.drop(columns=["Cabin"])                    # ~77% missing
df["Age"] = df["Age"].fillna(df["Age"].median())   # median is robust to outliers
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
print(df.isnull().sum())

**Why these methods:** `Age` has outliers and is skewed, so the median is safer than the mean. `Embarked` is categorical, so the mode is used. `Cabin` is about 77% empty, so it is dropped instead of filled. Rows with missing values were not deleted, to avoid losing data.

`Name` and `Ticket` are identifiers with no predictive pattern, so they are dropped.

In [2]:
df = df.drop(columns=["Name", "Ticket"])

<class 'NameError'>: name 'df' is not defined

## 7. Encode Categorical Data
`Sex` and `Embarked` are nominal (no natural order), so one-hot encoding is used. `drop_first=True` removes one redundant column per feature.

In [3]:
print("Before encoding:")
print(df[["Sex", "Embarked"]].head())

Before encoding:


<class 'NameError'>: name 'df' is not defined

In [4]:
df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True, dtype=int)
df.head()

<class 'NameError'>: name 'df' is not defined

## 8. Feature Scaling
Scaling matters for algorithms that depend on distances or gradients (KNN, SVM, logistic regression, gradient descent), because a feature with a big range would dominate the others. `Age` and `Fare` are on very different scales (Age ≈ 0–80, Fare ≈ 0–512), so we apply **Standardization**. `PassengerId` is only an ID, `Survived` is the target, and the encoded columns are already 0/1, so they are **not** scaled.

In [5]:
from sklearn.preprocessing import StandardScaler

scale_cols = ["Age", "Fare"]
df[scale_cols] = StandardScaler().fit_transform(df[scale_cols])
print(df[scale_cols].describe().round(2))

<class 'NameError'>: name 'df' is not defined

## 9. Final Verification

In [ ]:
print("Final shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
df.sample(5, random_state=42)

## 10. Save the Prepared Dataset

In [ ]:
df.to_csv("titanic_prepared.csv", index=False)
print("Saved titanic_prepared.csv")

## 11. Conclusion
The Titanic data had missing values in `Age` (177), `Cabin` (687) and `Embarked` (2), and no duplicates. `Cabin` was dropped, `Age` was filled with the median and `Embarked` with the mode. `Name` and `Ticket` were removed, `Sex` and `Embarked` were one-hot encoded, and `Age` and `Fare` were standardized. The final dataset has no missing values or duplicates and is ready for model building.